In [ ]:
# Compact identity plot for two aligned sequences (Clustal format).
# Shows the whole alignment as 3 stacked horizontal tracks (rows) so the
# plot stays narrow. Each position is colored by status:
#   match      - light grey
#   mismatch   - red
#   indel/gap  - blue
# This makes small insertions/deletions and substitutions visible without
# printing individual bases.

library(seqinr)

plot_alignment_identity <- function(aln_file, n_rows = 3,
                                     seq_names = NULL,
                                     col_match = "grey85",
                                     col_mismatch = "red",
                                     col_indel = "black") {

  aln <- read.alignment(file = aln_file, format = "clustal")
  if (aln$nb != 2) stop("Expected exactly 2 sequences in the alignment.")

  s1 <- toupper(strsplit(aln$seq[[1]], "")[[1]])
  s2 <- toupper(strsplit(aln$seq[[2]], "")[[1]])
  n  <- length(s1)
  if (length(s2) != n) stop("Aligned sequences have different lengths.")

  # Classify each alignment column
  status <- ifelse(s1 == s2, 0L,                      # match
             ifelse(s1 == "-" | s2 == "-", 2L, 1L))    # indel vs mismatch

  if (is.null(seq_names)) seq_names <- aln$nam

  # Pad so length is divisible by n_rows
  row_len <- ceiling(n / n_rows)
  pad_len <- row_len * n_rows - n
  status_padded <- c(status, rep(NA, pad_len))
  m <- matrix(status_padded, nrow = n_rows, byrow = TRUE)

  cols <- c(col_match, col_mismatch, col_indel)

  par(mfrow = c(n_rows, 1), mar = c(2, 1, 1, 1))

  for (r in seq_len(n_rows)) {
    row_vals <- m[r, ]
    start_pos <- (r - 1) * row_len + 1
    end_pos <- min(r * row_len, n)

    plot(NA, xlim = c(0, row_len), ylim = c(0, 1),
         xlab = "", ylab = "", axes = FALSE,
         main = paste0(start_pos, " - ", end_pos), cex.main = 0.8)

    valid <- !is.na(row_vals)
    x <- which(valid)
    rect(x - 1, 0, x, 1,
         col = cols[row_vals[valid] + 1], border = NA)

    box()
  }

  par(mfrow = c(1, 1))
  legend("bottom", inset = -0.3, xpd = NA, horiz = TRUE,
         legend = c("match", "mismatch", "indel"),
         fill = cols, bty = "n", cex = 0.8)
}

# Example usage:
# plot_alignment_identity("alignment.aln", n_rows = 3)

In [ ]:
clustawl = "LR_DESIREE_PAPER/DOWNSTREAM_ANALYSIS/promotor_enrichment_input_sequences/peptidase_alignment.aln"
# set background color to white
par(bg = "white")
svg("alignment.svg", width = 6, height = 1)
plot_alignment_identity(clustawl, n_rows = 1)
dev.off()